In [ ]:
# ================================================================
# Notebook 03: ToN-IoT-rareclass-IDS Final Model
# TON-IoT True Multi-Class Rare-Class Detection
#
# Final Model:
# No MI, No PCA + LightGBM OVR + MITM-targeted KMeansSMOTE
# ================================================================

import pandas as pd
import numpy as np
from collections import Counter
import warnings
import time

warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_class_weight

from imblearn.over_sampling import KMeansSMOTE, SMOTE, BorderlineSMOTE

from lightgbm import LGBMClassifier

print("\nNotebook 03: ToN-IoT-rareclass-IDS Final Model | LightGBM OVR + MITM KMeansSMOTE\n")
start_time = time.time()

# ================================================================
# PARAMETERS
# ================================================================

RANDOM_STATE = 42

TRAIN_PATH = "/kaggle/input/datasets/gowr1arun/unsw-nb15-and-ton-iot/toniot_network_train_80.csv"
TEST_PATH = "/kaggle/input/datasets/gowr1arun/unsw-nb15-and-ton-iot/toniot_network_test_20.csv"

TARGET_COL = "type"

MITM_CLASS_NAME = "mitm"
MITM_RATIO = 0.70

THRESH_GRID = np.round(np.arange(0.05, 0.96, 0.05), 2)

# ================================================================
# 1. LOAD DATA
# ================================================================

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Train 80%: {train_df.shape} | Test 20%: {test_df.shape}")

# ================================================================
# 2. CLEAN + CONSISTENT ENCODE
# ================================================================

def clean_and_encode_train_test(train_df, test_df):
    leak_cols = [
        "src_ip",
        "dst_ip",
        "http_uri",
        "ssl_subject",
        "ssl_issuer",
        "timestamp",
        "date",
        "time",
    ]

    train = train_df.drop(
        columns=[c for c in leak_cols if c in train_df.columns],
        errors="ignore",
    ).copy()

    test = test_df.drop(
        columns=[c for c in leak_cols if c in test_df.columns],
        errors="ignore",
    ).copy()

    train.replace([np.inf, -np.inf], np.nan, inplace=True)
    test.replace([np.inf, -np.inf], np.nan, inplace=True)

    for col in train.columns:
        if col in ["label", "type"]:
            continue

        if train[col].dtype == "object":
            mode_value = train[col].mode()[0]

            train[col] = train[col].fillna(mode_value).astype(str)
            test[col] = test[col].fillna(mode_value).astype(str)

            categories = pd.Index(train[col].unique())
            mapping = {cat: idx for idx, cat in enumerate(categories)}

            train[col] = train[col].map(mapping).astype(int)
            test[col] = test[col].map(mapping).fillna(-1).astype(int)

        else:
            mean_value = train[col].mean()
            train[col] = train[col].fillna(mean_value)
            test[col] = test[col].fillna(mean_value)

    return train, test


train_df, test_df = clean_and_encode_train_test(train_df, test_df)

# ================================================================
# 3. TRUE MULTI-CLASS TARGET SETUP
# ================================================================

X_train_full = train_df.drop(columns=["label", "type"], errors="ignore")
X_test_full = test_df.drop(columns=["label", "type"], errors="ignore")

label_encoder = LabelEncoder()

y_train_full = label_encoder.fit_transform(train_df[TARGET_COL].astype(str))
y_test_full = label_encoder.transform(test_df[TARGET_COL].astype(str))

class_names = list(label_encoder.classes_)
n_classes = len(class_names)

mitm_idx = class_names.index(MITM_CLASS_NAME)

print("\nClass mapping:")
for idx, name in enumerate(class_names):
    print(f"{idx}: {name}")

print("\nClass distribution train:", Counter(y_train_full))
print("Class distribution test :", Counter(y_test_full))
print("\nMITM class index:", mitm_idx)

# ================================================================
# 4. FEATURE PROCESSING
# No MI, No PCA.
# Only log transform + scaling.
# ================================================================

def log_transform(X):
    return np.log1p(np.abs(X))


# Train/calibration split for threshold tuning
sss = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

final_tr_idx, calib_idx = next(sss.split(X_train_full, y_train_full))

X_final_raw = X_train_full.iloc[final_tr_idx]
y_final = y_train_full[final_tr_idx]

X_calib_raw = X_train_full.iloc[calib_idx]
y_calib = y_train_full[calib_idx]

print("\nFinal train distribution:", Counter(y_final))
print("Calibration distribution:", Counter(y_calib))
print("Official test distribution:", Counter(y_test_full))

# Log transform
X_final_log = log_transform(X_final_raw)
X_calib_log = log_transform(X_calib_raw)
X_test_log = log_transform(X_test_full)

# Scale
scaler = StandardScaler()

X_final_feat = scaler.fit_transform(X_final_log)
X_calib_feat = scaler.transform(X_calib_log)
X_test_feat = scaler.transform(X_test_log)

print("\nFeature count after No MI / No PCA:", X_final_feat.shape[1])

# ================================================================
# 5. MITM-TARGETED KMeansSMOTE
# ================================================================

def mitm_targeted_resample(X, y, mitm_idx, mitm_ratio=0.70):
    counts = Counter(y)
    max_count = max(counts.values())

    target_mitm_count = int(max_count * mitm_ratio)

    if counts[mitm_idx] >= target_mitm_count:
        print("MITM already above target count. No oversampling applied.")
        return X, y

    sampling_strategy = {mitm_idx: target_mitm_count}

    print("\nMITM sampling strategy:")
    print({class_names[k]: v for k, v in sampling_strategy.items()})

    samplers = [
        (
            "KMeansSMOTE",
            KMeansSMOTE(
                sampling_strategy=sampling_strategy,
                random_state=RANDOM_STATE,
                k_neighbors=3,
                cluster_balance_threshold=0.01,
            ),
        ),
        (
            "SMOTE",
            SMOTE(
                sampling_strategy=sampling_strategy,
                random_state=RANDOM_STATE,
                k_neighbors=3,
            ),
        ),
        (
            "BorderlineSMOTE",
            BorderlineSMOTE(
                sampling_strategy=sampling_strategy,
                random_state=RANDOM_STATE,
                k_neighbors=3,
            ),
        ),
    ]

    for name, sampler in samplers:
        try:
            X_res, y_res = sampler.fit_resample(X, y)
            print(f"{name} worked.")
            print("Before sampling:", Counter(y))
            print("After sampling :", Counter(y_res))
            return X_res, y_res
        except Exception as e:
            print(f"{name} failed: {str(e)[:150]}")

    print("All samplers failed. Continuing without resampling.")
    return X, y


X_final_res, y_final_res = mitm_targeted_resample(
    X_final_feat,
    y_final,
    mitm_idx=mitm_idx,
    mitm_ratio=MITM_RATIO,
)

# ================================================================
# 6. LIGHTGBM OVR MODEL
# ================================================================

def make_lightgbm_binary_model():
    return LGBMClassifier(
        n_estimators=350,
        max_depth=-1,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2.0,
        reg_alpha=0.2,
        objective="binary",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    )


def train_ovr_lightgbm(X, y, n_classes, mitm_idx):
    models = []

    original_counts = Counter(y)
    max_count = max(original_counts.values())

    for c in range(n_classes):
        print(f"Training OVR classifier for class {c}: {class_names[c]}")

        y_bin = (y == c).astype(int)

        classes_bin = np.array([0, 1])
        weights_bin = compute_class_weight(
            class_weight="balanced",
            classes=classes_bin,
            y=y_bin,
        )

        weight_map = {
            0: weights_bin[0],
            1: weights_bin[1],
        }

        sample_weight = np.array([weight_map[v] for v in y_bin])

        cls_count = original_counts[c]
        rare_multiplier = np.sqrt(max_count / cls_count)
        sample_weight[y_bin == 1] *= rare_multiplier

        if c == mitm_idx:
            sample_weight[y_bin == 1] *= 1.5

        model = make_lightgbm_binary_model()
        model.fit(X, y_bin, sample_weight=sample_weight)

        models.append(model)

    return models


def predict_ovr_proba(models, X):
    probs = []

    for model in models:
        probs.append(model.predict_proba(X)[:, 1])

    return np.vstack(probs).T


models = train_ovr_lightgbm(
    X_final_res,
    y_final_res,
    n_classes=n_classes,
    mitm_idx=mitm_idx,
)

# ================================================================
# 7. PER-CLASS THRESHOLD TUNING
# ================================================================

def tune_thresholds(y_val, proba_val, n_classes):
    thresholds = np.ones(n_classes) * 0.50

    for c in range(n_classes):
        y_true_bin = (y_val == c).astype(int)

        best_t = 0.50
        best_f1 = -1

        for t in THRESH_GRID:
            y_pred_bin = (proba_val[:, c] >= t).astype(int)
            score = f1_score(y_true_bin, y_pred_bin, zero_division=0)

            if score > best_f1:
                best_f1 = score
                best_t = t

        thresholds[c] = best_t

    return thresholds


def predict_with_thresholds(proba, thresholds):
    adjusted = proba / thresholds.reshape(1, -1)
    return np.argmax(adjusted, axis=1)


proba_calib = predict_ovr_proba(models, X_calib_feat)
thresholds = tune_thresholds(y_calib, proba_calib, n_classes)

print("\nFinal per-class thresholds:")
print({class_names[i]: float(thresholds[i]) for i in range(n_classes)})

# ================================================================
# 8. OFFICIAL TEST EVALUATION
# ================================================================

proba_test = predict_ovr_proba(models, X_test_feat)
y_test_pred = predict_with_thresholds(proba_test, thresholds)

def rare_class_metrics(y_true, y_pred, class_idx):
    y_true_bin = (y_true == class_idx).astype(int)
    y_pred_bin = (y_pred == class_idx).astype(int)

    return {
        "precision": precision_score(y_true_bin, y_pred_bin, zero_division=0),
        "recall": recall_score(y_true_bin, y_pred_bin, zero_division=0),
        "f1": f1_score(y_true_bin, y_pred_bin, zero_division=0),
    }


test_acc = accuracy_score(y_test_full, y_test_pred)
test_bal_acc = balanced_accuracy_score(y_test_full, y_test_pred)
test_macro_prec = precision_score(y_test_full, y_test_pred, average="macro", zero_division=0)
test_macro_rec = recall_score(y_test_full, y_test_pred, average="macro", zero_division=0)
test_macro_f1 = f1_score(y_test_full, y_test_pred, average="macro", zero_division=0)
test_weighted_f1 = f1_score(y_test_full, y_test_pred, average="weighted", zero_division=0)

try:
    y_test_oh = pd.get_dummies(y_test_full)
    test_auc = roc_auc_score(
        y_test_oh,
        proba_test,
        average="macro",
        multi_class="ovr",
    )
except Exception:
    test_auc = np.nan

mitm_scores = rare_class_metrics(y_test_full, y_test_pred, mitm_idx)

print("\n" + "=" * 90)
print("PAPER-READY RESULT SUMMARY")
print("=" * 90)

print("Model              : D. No MI, no PCA + LightGBM OVR + MITM KMeansSMOTE")
print(f"Accuracy           : {test_acc:.6f}")
print(f"Weighted F1        : {test_weighted_f1:.6f}")
print(f"Macro F1           : {test_macro_f1:.6f}")
print(f"Macro Recall       : {test_macro_rec:.6f}")
print(f"Balanced Accuracy  : {test_bal_acc:.6f}")
print(f"Macro AUC OVR      : {test_auc:.6f}")
print(f"MITM Precision     : {mitm_scores['precision']:.6f}")
print(f"MITM Recall        : {mitm_scores['recall']:.6f}")
print(f"MITM F1            : {mitm_scores['f1']:.6f}")
print(f"Number of Features : {X_final_feat.shape[1]}")

print("\nPer-class report:")
print(
    classification_report(
        y_test_full,
        y_test_pred,
        target_names=class_names,
        digits=4,
        zero_division=0,
    )
)

cm = confusion_matrix(y_test_full, y_test_pred)

print("\nConfusion matrix:")
print(cm)

# ================================================================
# 9. SAVE OUTPUTS
# ================================================================

summary_df = pd.DataFrame([
    {
        "Model": "D. No MI, no PCA + LightGBM OVR + MITM KMeansSMOTE",
        "Dataset": "TON-IoT",
        "Task": "True multi-class using type column",
        "Feature Strategy": "Log transform + StandardScaler; No MI; No PCA",
        "Sampling": "MITM-targeted KMeansSMOTE",
        "Classifier": "One-vs-Rest LightGBM",
        "Accuracy": test_acc,
        "Balanced Accuracy": test_bal_acc,
        "Macro Precision": test_macro_prec,
        "Macro Recall": test_macro_rec,
        "Macro F1": test_macro_f1,
        "Weighted F1": test_weighted_f1,
        "Macro AUC OVR": test_auc,
        "MITM Precision": mitm_scores["precision"],
        "MITM Recall": mitm_scores["recall"],
        "MITM F1": mitm_scores["f1"],
        "Number of Features": X_final_feat.shape[1],
        "MITM Sampling Ratio": MITM_RATIO,
    }
])

summary_path = "/kaggle/working/final_model_d_summary.csv"
summary_df.to_csv(summary_path, index=False)

report_dict = classification_report(
    y_test_full,
    y_test_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report_dict).transpose()

report_path = "/kaggle/working/final_model_d_per_class_report.csv"
report_df.to_csv(report_path)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{c}" for c in class_names],
    columns=[f"pred_{c}" for c in class_names],
)

cm_path = "/kaggle/working/final_model_d_confusion_matrix.csv"
cm_df.to_csv(cm_path)

print("\nSaved files:")
print(summary_path)
print(report_path)
print(cm_path)

print(f"\nNotebook 03 complete in {(time.time() - start_time) / 60:.2f} minutes.")

print("""
Notebook 03 conclusion:
The final ToN-IoT-rareclass-IDS model uses true multi-class labels from the `type` column.
The selected model is LightGBM in a One-vs-Rest setup with MITM-targeted KMeansSMOTE and per-class threshold tuning.
This notebook produces the main final result files used in the paper and GitHub repository.
""")